In [12]:
import pandas as pd
df = pd.read_csv('loan_default.csv')
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [ ]:
#Removing duplicate coloumns
duplicate_cols = df.columns[df.T.duplicated()]

print("Duplicate columns:", duplicate_cols)

df = df.loc[:, ~df.T.duplicated()]

In [13]:
#Remove Unecessary coloumn
df = df.drop(columns='LoanID')

In [14]:
X = df.drop('Default', axis=1)
y = df['Default']

In [15]:
#Encoding categorical variables
X = pd.get_dummies(X, drop_first=True)

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [17]:
#Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [18]:
#Baseline Model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Baseline Accuracy:", accuracy)

Baseline Accuracy: 0.885275112590562


***1.Exhaustive Feature Selection***

In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from mlxtend.feature_selection import ExhaustiveFeatureSelector

# Load dataset
df = pd.read_csv('loan_default.csv')

# Remove unnecessary column
df = df.drop(columns='LoanID')

# Remove duplicate columns-->Will Take Time
#df = df.loc[:, ~df.T.duplicated()]

df = df.dropna(subset=['Default'])

# Sample data (important for speed)
df = df.sample(5000, random_state=42)

# Separate features and target
X = df.drop('Default', axis=1)
y = df['Default']

# Encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert back to dataframe
X_train = pd.DataFrame(X_train, columns=X.columns)
X_test = pd.DataFrame(X_test, columns=X.columns)

# Logistic Regression model
lr = LogisticRegression(max_iter=1000)

# Exhaustive Feature Selector
efs = ExhaustiveFeatureSelector(
    lr,
    min_features=1,
    max_features=3,
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)

# Fit EFS
efs.fit(X_train, y_train)

# Selected features
selected_features = list(efs.best_feature_names_)

print("Selected Features:")
print(selected_features)

# Reduced datasets
X_train_efs = X_train[selected_features]
X_test_efs = X_test[selected_features]

# Train model
lr.fit(X_train_efs, y_train)

# Prediction
y_pred = lr.predict(X_test_efs)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("EFS Accuracy:", accuracy)

Features: 2324/2324

Selected Features:
['Age', 'Income', 'InterestRate']
EFS Accuracy: 0.889


**2.Sequential Forward Selection**

In [48]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SequentialFeatureSelector

# Load dataset
df = pd.read_csv('loan_default.csv')

# Remove unnecessary column
df = df.drop(columns='LoanID')

# Remove duplicate columns-->Will Take Time
#df = df.loc[:, ~df.T.duplicated()]

df = df.dropna(subset=['Default'])

# Separate features and target
X = df.drop('Default', axis=1)
y = df['Default']

# Encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert back to dataframe
X_train = pd.DataFrame(X_train, columns=X.columns)
X_test = pd.DataFrame(X_test, columns=X.columns)

# Logistic Regression model
lr = LogisticRegression(max_iter=1000)

# Sequential Forward Selection
sfs = SequentialFeatureSelector(
    lr,
    n_features_to_select=3,
    direction='forward',
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)

# Fit SFS
sfs.fit(X_train, y_train)

# Selected features
selected_features = X_train.columns[sfs.get_support()]

print("Selected Features:")
print(selected_features)

# Reduced datasets
X_train_sfs = X_train[selected_features]
X_test_sfs = X_test[selected_features]

# Train model
lr.fit(X_train_sfs, y_train)

# Prediction
y_pred = lr.predict(X_test_sfs)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("SFS Accuracy:", accuracy)

Selected Features:
Index(['Age', 'Income', 'LoanAmount'], dtype='object')
SFS Accuracy: 0.8838652829449775


***3.Sqeuantial Backward Elimination***

In [49]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SequentialFeatureSelector

# Load dataset
df = pd.read_csv('loan_default.csv')

# Remove unnecessary column
df = df.drop(columns='LoanID')

# Remove duplicate columns-->Will Take Time
#df = df.loc[:, ~df.T.duplicated()]

df = df.dropna(subset=['Default'])

# Separate features and target
X = df.drop('Default', axis=1)
y = df['Default']

# Encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert back to dataframe
X_train = pd.DataFrame(X_train, columns=X.columns)
X_test = pd.DataFrame(X_test, columns=X.columns)

# Logistic Regression model
lr = LogisticRegression(max_iter=1000)

# Sequential Backward Selection
sbs = SequentialFeatureSelector(
    lr,
    n_features_to_select=3,
    direction='backward',
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)

# Fit SBS
sbs.fit(X_train, y_train)

# Selected features
selected_features = X_train.columns[sbs.get_support()]

print("Selected Features:")
print(selected_features)

# Reduced datasets
X_train_sbs = X_train[selected_features]
X_test_sbs = X_test[selected_features]

# Train model
lr.fit(X_train_sbs, y_train)

# Prediction
y_pred = lr.predict(X_test_sbs)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("SBS Accuracy:", accuracy)

Selected Features:
Index(['Income', 'MonthsEmployed', 'InterestRate'], dtype='object')
SBS Accuracy: 0.8838652829449775


***4A.Recursive Feature Elimination***

In [50]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFE

# Load dataset
df = pd.read_csv('loan_default.csv')

# Remove unnecessary column
df = df.drop(columns='LoanID')

# Remove duplicate columns-->Will Take Time
#df = df.loc[:, ~df.T.duplicated()]

# Separate features and target
X = df.drop('Default', axis=1)
y = df['Default']

# Encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert back to dataframe
X_train = pd.DataFrame(X_train, columns=X.columns)
X_test = pd.DataFrame(X_test, columns=X.columns)

# Logistic Regression model
lr = LogisticRegression(max_iter=1000)

# RFE
rfe = RFE(
    estimator=lr,
    n_features_to_select=3
)

# Fit RFE
rfe.fit(X_train, y_train)

# Selected features
selected_features = X_train.columns[rfe.support_]

print("Selected Features:")
print(selected_features)

# Reduced datasets
X_train_rfe = X_train[selected_features]
X_test_rfe = X_test[selected_features]

# Train model
lr.fit(X_train_rfe, y_train)

# Prediction
y_pred = lr.predict(X_test_rfe)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("RFE Accuracy:", accuracy)

Selected Features:
Index(['Age', 'Income', 'InterestRate'], dtype='object')
RFE Accuracy: 0.8838652829449775


4B.Recursive Feature Elimination With Cross Validation

In [53]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFECV

# Load dataset
df = pd.read_csv('loan_default.csv')

# Remove unnecessary column
df = df.drop(columns='LoanID')

# Remove duplicate columns
#df = df.loc[:, ~df.T.duplicated()]

# Separate features and target
X = df.drop('Default', axis=1)
y = df['Default']

# Encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert back to dataframe
X_train = pd.DataFrame(X_train, columns=X.columns)
X_test = pd.DataFrame(X_test, columns=X.columns)

# Logistic Regression model
lr = LogisticRegression(max_iter=1000)

# RFECV
rfecv = RFECV(
    estimator=lr,
    step=1,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

# Fit RFECV
rfecv.fit(X_train, y_train)

# Optimal number of features
print("Optimal number of features:", rfecv.n_features_)

# Selected features
selected_features = X_train.columns[rfecv.support_]

print("\nSelected Features:")
print(selected_features)

# Transform datasets
X_train_rfecv = rfecv.transform(X_train)
X_test_rfecv = rfecv.transform(X_test)

# Train final model
lr.fit(X_train_rfecv, y_train)

# Prediction
y_pred = lr.predict(X_test_rfecv)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("\nRFECV Accuracy:", accuracy)

Optimal number of features: 9

Selected Features:
Index(['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
       'InterestRate', 'EmploymentType_Unemployed', 'HasDependents_Yes',
       'HasCoSigner_Yes'],
      dtype='object')

RFECV Accuracy: 0.8850205600156648


# Observation on Wrapper Feature Selection Methods

Different wrapper feature selection methods produced different subsets of features because each method follows a different search and optimization strategy.

Exhaustive Feature Selection (EFS) checks all possible feature combinations and therefore provides the globally best subset among the searched combinations. It selected:

* Age
* Income
* InterestRate

and achieved the highest accuracy of 0.889.

Sequential Forward Selection (SFS) starts with zero features and adds features one-by-one based on local improvement in model performance. It selected:

* Age
* Income
* LoanAmount

with accuracy 0.8838.

Sequential Backward Selection (SBS) starts with all features and removes the least useful features iteratively. It selected:

* Income
* MonthsEmployed
* InterestRate

with accuracy 0.8838.

Recursive Feature Elimination (RFE) recursively removes features based on model coefficient importance. It selected:

* Age
* Income
* InterestRate

with accuracy 0.8838.

RFECV (Recursive Feature Elimination with Cross Validation) automatically determines the optimal number of features using cross-validation. It selected 9 features and achieved accuracy 0.8850.

The differences in selected features occur because:

1. Different methods search the feature space differently.
2. Some methods are greedy (SFS/SBS), while others use recursive elimination (RFE/RFECV).
3. Many features in the dataset contain overlapping or correlated information.
4. Multiple feature subsets can provide similar predictive performance.

The results indicate that features such as Age, Income, and InterestRate are consistently strong predictors of loan default. Even after reducing the number of features significantly, the model maintained accuracy close to the baseline accuracy of 0.88, showing that only a small subset of features carries most of the predictive information.
